# Part 8 · Build an agent, ship it, then fix its tool layer

**Six steps, about twelve minutes.** A developer builds an agent against GitHub,
AgentRegistry ships it to kagent, and then one field on the gateway takes the same
job from twenty model round trips down to two. It ends with the agent being told to
merge a pull request and not being able to.

| | Step | Roughly |
|---|---|---|
| 0 | Seed the frozen pull requests (once, not on stage) | 2 min |
| 1 | What one MCP server costs you | 1 min |
| 2 | The gateway holds the credential | 1 min |
| 3 | Scaffold the agent from the approved catalogue | 3 min |
| 4 | Deploy it with AgentRegistry, run it in the kagent UI | 3 min |
| 5 | One field: 20 round trips become 2 | 2 min |
| 6 | Take the write tools away | 2 min |

Runs on **`mesh1`** and needs the Part 4 platform
(`demo-scripts/agentregistry/setup-mesh1.sh`) plus a GitHub PAT.

**The data is frozen on purpose.** Reading live pull requests from a busy upstream repo
is a bad bet on a projector: the answer changes hour to hour, the numbers stop matching
your slides, and the model sometimes over-fetches and trips code mode's call cap. Step 0
seeds a repo you own with exactly twenty four open pull requests in known states, so
the report is the same every time and "all open pull requests" is always twenty four.

There is an appendix at the end covering the other two `toolMode` settings. Skip it in
a short slot.

## 0. Seed the frozen pull requests

Run this **once**, well before you present. It creates twenty four open pull requests
in a repo you own: two signed off, three drafts, four held, and fifteen waiting on a
sign-off.

Twenty four rather than a handful for a reason. At that size the default mode needs
nineteen separate calls, which is a trace you can scroll for ten seconds on a
projector, and the difference stops being a rounding error.

The gate is built from three things a personal access token can genuinely produce, and
all three are ordinary in real repositories:

| verdict | how it is produced |
|---|---|
| `draft` | the pull request is opened as a draft |
| `on hold` | the `do-not-merge/hold` label, the Prow convention half the Kubernetes ecosystem uses |
| `no sign-off` | no comment **starting** with `LGTM` |
| `READY` | none of the above |

One pull request carries a comment that says "lgtm" in the middle of a sentence while
explicitly declining to sign off, because the rule is `LGTM` at the **start** of a
comment. It is there to catch a report that skim-read the rule, and it has held across
every run.

Two things it deliberately does **not** use. GitHub will not let you approve your own
pull request, and a token cannot write check runs or commit statuses at all, both 403.
`mergeable` was tried and dropped: GitHub computes it lazily, so it returns `null`
often enough to make a live demo unreliable. A label always reads the same.

In [ ]:
# once, not on stage. RESEED=1 closes the demo PRs and starts over.
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
REPO="${DEMO_REPO:-tjorourke/kagent}" ./demo-scripts/prtriage/scripts/seed-demo-repo.sh

## Setup and Connect

In [ ]:
# from the folder that holds this notebook (the one containing demo-scripts/)
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh >/dev/null 2>&1
./demo-scripts/prtriage/scripts/setup.sh

In [ ]:
LAB=$PWD
until [ -d "$LAB/demo-scripts/prtriage" ]; do
  [ "$LAB" = / ] && { echo "✗ run this from the demo suite folder (the one with demo-scripts/)"; break; }
  LAB=$(dirname "$LAB")
done
cd "$LAB"
set -a; . "${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}" 2>/dev/null; set +a
source demo-scripts/agentregistry/scripts/connect.sh
export PART8=demo-scripts/prtriage
export LB=$(kubectl --context kind-mesh1 -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
export MCP="http://github-mcp.${LB}.sslip.io/"
export ASK="demo-scripts/agentregistry/scripts/ask.sh"
export KC="kubectl --context kind-mesh1"
# The demo reads FROZEN pull requests from a repo you own, so the report is identical
# every run. See scripts/seed-demo-repo.sh for how they are built and why.
export DEMO_REPO="${DEMO_REPO:-tjorourke/kagent}"
echo
printf "  %-20s %s\n" "kagent UI:"        "http://${KAGENT_UI_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "AgentRegistry UI:" "http://${AR_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "MCP endpoint:"     "$MCP"
printf "  %-20s %s\n" "demo repo:"        "https://github.com/${DEMO_REPO}/pulls"

`mcp.sh` is a few lines of curl: initialize, keep the session id, send one JSON-RPC
call. Used in steps 1 and 6 so you can see the wire rather than a framework's idea
of it.

In [ ]:
cat > /tmp/mcp.sh <<'SH'
#!/usr/bin/env bash
# mcp.sh <endpoint> <method> [params-json] — one MCP call, session handled.
#
# The endpoint is a *.<ip>.sslip.io name, which means the address is already in the
# name. We pull it out and hand it to curl with --resolve rather than asking a
# resolver, because plenty of home-router and ISP resolvers refuse to return a
# private address for a public name (DNS rebinding protection) and the failure looks
# like the gateway being down.
set -euo pipefail
EP="$1"; METHOD="$2"; PARAMS="${3:-}"
HOST=$(printf '%s' "$EP" | sed -E 's#^https?://##; s#[:/].*$##')
IP=$(printf '%s' "$HOST" | sed -nE 's#.*[.]?([0-9]{1,3}-[0-9]{1,3}-[0-9]{1,3}-[0-9]{1,3})[.]sslip[.]io$#\1#p' | tr '-' '.')
[ -n "$IP" ] || IP=$(printf '%s' "$HOST" | sed -nE 's#.*[.]([0-9]{1,3}[.][0-9]{1,3}[.][0-9]{1,3}[.][0-9]{1,3})[.]sslip[.]io$#\1#p')
PORT=$(printf '%s' "$EP" | sed -nE 's#^https?://[^:/]+:([0-9]+).*#\1#p'); PORT=${PORT:-80}
RESOLVE=(); [ -n "$IP" ] && RESOLVE=(--resolve "$HOST:$PORT:$IP")
H=(-H "Content-Type: application/json" -H "Accept: application/json, text/event-stream")
HDR=$(mktemp)
curl -s -m 60 -X POST "$EP" "${RESOLVE[@]}" "${H[@]}" -D "$HDR" -o /dev/null \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"demo8","version":"1"}}}'
SID=$(grep -i '^mcp-session-id:' "$HDR" | tr -d '\r' | awk '{print $2}')
BODY=$(python3 -c 'import json,sys;print(json.dumps({"jsonrpc":"2.0","id":2,"method":sys.argv[1],"params":json.loads(sys.argv[2] or "{}")}))' "$METHOD" "$PARAMS")
curl -s -m 180 -X POST "$EP" "${RESOLVE[@]}" "${H[@]}" ${SID:+-H "Mcp-Session-Id: $SID"} -d "$BODY" \
  | sed 's/^data: //' | grep -v '^event:' | grep -v '^$'
SH
chmod +x /tmp/mcp.sh; echo "wrote /tmp/mcp.sh"

## 1. What one MCP server costs you

GitHub's hosted MCP server. One server. This is what it puts in the model's context on
every single turn, and what it lets the agent do.

In [ ]:
/tmp/mcp.sh "$MCP" tools/list > /tmp/tools-standard.json
python3 - /tmp/tools-standard.json <<'PY'
import json,sys
d=json.load(open(sys.argv[1])); t=d["result"]["tools"]
WRITE=("create_","update_","delete_","merge_","push_","add_","fork_","_write","request_copilot_review")
w=sorted(x["name"] for x in t if any(k in x["name"] for k in WRITE))
print("tools:                %d" % len(t))
print("tools/list payload:   %d bytes" % len(json.dumps(d)))
print("of which can write:   %d" % len(w))
print()
print("the write tools your agent just acquired:")
print("  " + ", ".join(w))
PY

And the token cost, from Anthropic's own counter rather than an estimate:

In [ ]:
python3 - /tmp/tools-standard.json <<'PY' > /tmp/count.json
import json,sys
d=json.load(open(sys.argv[1]))
tools=[{"name":t["name"],"description":t.get("description",""),
        "input_schema":t.get("inputSchema",{"type":"object"})} for t in d["result"]["tools"]]
print(json.dumps({"model":"claude-sonnet-4-5","tools":tools,"messages":[{"role":"user","content":"hi"}]}))
PY
curl -s https://api.anthropic.com/v1/messages/count_tokens \
  -H "x-api-key: $ANTHROPIC_API_KEY" -H "anthropic-version: 2023-06-01" \
  -H "content-type: application/json" -d @/tmp/count.json

**14,572 tokens before anyone types anything, and 17 of the 44 tools can write**:
`delete_file`, `push_files`, `merge_pull_request`. A token scope cannot say "this one
agent may only read pull requests", because a scope is coarse and the token is shared
by everyone who uses it.

Neither problem is fixable in the agent's code. Both are fixable one layer down.

## 2. The gateway holds the credential

`setup.sh` applied this already. Two fields carry the idea: `protocol: StreamableHTTP`
reaches GitHub's hosted server, and `policies.auth.secretRef` injects the PAT upstream
from a Secret the gateway reads.

In [ ]:
sed -n '/^apiVersion/,$p' $PART8/yaml/10-github-backend.yaml | sed '/^---$/,$d'

Proof rather than assertion. This call sends **no `Authorization` header**:

In [ ]:
echo "  == the client sends Content-Type and Accept, and nothing else =="
/tmp/mcp.sh "$MCP" tools/call \
  '{"name":"list_pull_requests","arguments":{"owner":"kagent-dev","repo":"kagent","state":"open","perPage":3,"fields":["number","title"]}}' \
  | python3 -c "
import json,sys
for r in json.loads(json.load(sys.stdin)['result']['content'][0]['text']):
    print('  #%s  %s' % (r['number'], r['title'][:62]))
"

The agent it is about to serve holds no GitHub credential at all, so it cannot leak
one. That is the first reason to put a gateway here.

## 3. Scaffold the agent from the approved catalogue

Two things are in the catalogue, and both matter.

The **approved MCP server** points at the gateway route, not at `api.githubcopilot.com`.
A developer picking GitHub out of the catalogue gets GitHub through the enforcement
point, and there is no catalogue entry meaning "GitHub, but skip the gateway".

The **approved skill** is the platform team's write-up of how to drive this gateway:
the parameter is `pullNumber` not `pull_number`, `get_check_runs` returns an object
while `get_reviews` returns an array, the code sandbox has no `Date`, a program gets
at most 20 upstream calls. Every line is there because a run failed without it. The
team learns this once.

In [ ]:
echo "  == the approved MCP server: note the URL =="
arctl get mcpserver github-mcp -o yaml | sed -n '/^spec:/,$p' | grep -E "url:|title:"
echo
echo "  == the approved skill =="
arctl get skill release-report -o yaml | sed -n '/^spec:/,$p' | grep -E "title:|description:" | head -3

Now scaffold. `arctl init` writes a complete, runnable ADK + Python project; `--mcp
github-mcp@latest` wires it to the approved server. The scaffold ships two sample
tools that roll dice, and we swap them for one local tool, `today`, which exists
because the gateway's code sandbox has no clock.

In [ ]:
FORCE=1 $PART8/scripts/scaffold-agent.sh

In [ ]:
echo "  == the agent's only local tool =="
sed -n '/^def today/,/^    return/p' prtriage/prtriage/agent.py | head -4
echo
echo "  == and what it was wired to =="
grep -A4 mcpServers prtriage/agent.yaml

Then bake the approved skill into the agent's prompts and build. `rebuild-agent.sh`
does that, and works around a trap worth an hour of anyone's day: the image is
`localhost:5001/prtriage:latest` and kagent runs it `IfNotPresent`, so pushing a new
image on the same tag and restarting quietly reuses the node's cached copy. It drops
the cached image from every node and then checks the running pod really has the
current skill.

In [ ]:
$PART8/scripts/rebuild-agent.sh

## 4. Deploy it with AgentRegistry, and run it

One `Deployment` record names the agent and the runtime. AgentRegistry does the rest:
it creates the kagent `Agent`, derives the MCP wiring from the catalogue, and the
kagent controller brings it up. No Helm, no kubectl apply of a pod spec.

In [ ]:
cat $PART8/yaml/40-deploy-kagent.yaml

In [ ]:
arctl apply -f $PART8/yaml/40-deploy-kagent.yaml
until $KC -n kagent get deploy/prtriage >/dev/null 2>&1; do sleep 2; done
$KC -n kagent rollout status deploy/prtriage --timeout=240s
echo
$KC -n kagent get agent prtriage
$KC -n kagent get pods | grep -E 'NAME|prtriage'

### Run it

**On stage, do this in the kagent UI**: open `http://<kagent-ui-host>` from the
Connect cell, pick **prtriage**, and paste the question. You get the answer and the
tool-call span tree in the Tracing tab, which is the thing worth projecting.

The cell below is the same call headless, and it prints the tool-call trace itself, so
the notebook stands alone. Note the mode we are in first.

In [ ]:
$KC -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}'
$PART8/scripts/reload-agent.sh

In [ ]:
AGENT_PREFIX=prtriage $ASK "Give me the release report for $DEMO_REPO, all open pull requests." \
  | tee /tmp/run-standard.txt | tail -20
echo
echo "  == what that cost =="
$PART8/scripts/trace-cost.sh /tmp/run-standard.txt

Read the trace, not just the answer. **Nineteen round trips and 75,588 bytes.** One call to list the pull requests, then one
per pull request to read its discussion, each re-sending the whole conversation so far,
with every raw API response landing in the context window on the way.

Nineteen rather than twenty five, because the approved skill tells it not to fetch what
cannot change the answer: a draft or a held pull request is already decided, so reading
its comments buys nothing. The registry saves seven calls before the gateway does
anything.

Now look at the last line of the report. It says **Scanned: 25**. There are twenty
four. It read them one at a time and lost count, in both of the runs recorded here.

In [ ]:
echo "  == the report said these are ready to merge =="
READY=$(grep -oE "^Ready to merge: .*" /tmp/run-standard.txt | sed 's/Ready to merge: //; s/#//g; s/,//g')
echo "  $READY"
echo
echo "  == what GitHub says about their reviews =="
for pr in $READY; do
  printf "  PR #%-6s " "$pr"
  /tmp/mcp.sh "$MCP" tools/call \
    "{\"name\":\"pull_request_read\",\"arguments\":{\"owner\":\"kagent-dev\",\"repo\":\"kagent\",\"pullNumber\":$pr,\"method\":\"get_reviews\"}}" \
  | python3 -c "
import json,sys
rv=json.loads(json.load(sys.stdin)['result']['content'][0]['text'])
print('no reviews at all' if not rv else ', '.join(r.get('state','?') for r in rv))
"
done

If that list came back with no approvals, the report was **wrong**, and it is wrong
reproducibly: two runs in a row on this cluster reported four and then five
unapproved pull requests as ready to merge.

Nothing is misconfigured, and the model is not being stupid. It made twenty separate
calls, and by the time it came to write the report the review data was tens of
thousands of tokens behind it, buried under check-run JSON and Copilot review bodies.
It lost track.

So the default mode is not merely slow. **It is less accurate**, and it is wrong in the
worst possible direction: it tells you to ship things nobody approved.

## 5. One field: 20 round trips become 2

`toolMode: CodeSearch` stops the gateway handing the model 44 tools. It gives it two
instead: `get_tool` to look up an operation's schema, and `run_code` to execute a
JavaScript program against them. The model writes one program, the gateway runs it in
a sandbox, makes the upstream calls, and returns only what the program returns.

Same agent. Same image. Same catalogue. One field on the backend.

(`toolMode: Code` is the conservative sibling: one `run_code` tool with all 44
signatures baked into its description, so nothing has to be looked up. It costs 6,302
schema tokens against CodeSearch's 1,300 and lands on the same two round trips. Use it
if you would rather the model never have to ask. The appendix has both.)

In [ ]:
$KC -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"CodeSearch"}}}'
sleep 8
/tmp/mcp.sh "$MCP" tools/list | python3 -c "
import json,sys
t=json.load(sys.stdin)['result']['tools']
print('tools the model now gets:',len(t),'->',', '.join(x['name'] for x in t))
"
$PART8/scripts/reload-agent.sh

In [ ]:
AGENT_PREFIX=prtriage $ASK "Give me the release report for $DEMO_REPO, all open pull requests." \
  | tee /tmp/run-code.txt | tail -22
echo
echo "  == what that cost =="
$PART8/scripts/trace-cost.sh /tmp/run-code.txt

**Two round trips.** `today()`, then one program that made nine GitHub calls inside the
gateway and returned the finished report, from **11 bytes** of intermediate data
through the model instead of 6,747.

The answer is identical to step 4, and it stays identical run after run, because the
filtering happened in the sandbox on data the model never had to hold or remember.
There is nothing left to lose track of, which is precisely why the accuracy problem
above disappears.

Measured on this cluster, twenty four pull requests, two runs each and identical both
times:

| | Standard | CodeSearch |
|---|---|---|
| tools the model holds | 45 | 2 |
| schema tokens per turn | 14,572 | **1,300** |
| model round trips | 19 | **2** |
| payload through the model | 75,588 B | **10 B** |
| counted the pull requests correctly | no, said 25 | **yes, 24** |

**Do not claim it is faster.** Both land around thirty seconds, because code mode
spends its saving on the model writing the program, and somebody in the room will time
you. The claims that hold every single run are the round trips, the bytes and the
count.

**What to actually show on the projector:** scroll the Standard trace, which is a
screenful of raw GitHub JSON, then show the CodeSearch trace, which is two lines. That
contrast reads from the back of the room in a way a stopwatch never would, and the byte
count under each makes it a number rather than an impression.

It gets worse than a miscount when the responses are bigger. Against the live upstream
repo, where every pull request drags in check-run JSON and full review bodies, the same
question took twenty round trips and 87,630 bytes, and twice reported pull requests as
ready to merge that had no approval at all. Once it dropped a pull request and reported
seven of eight. Show that as evidence rather than staging it: it is stochastic, and a
demo must not depend on the model making a mistake.

The sandbox is deliberately small, and it is worth one cell to show why that matters:
the only way out is the generated tool functions. There is no `fetch`, no `require`,
no `process`. A program cannot phone home, it can only call approved tools.

In [ ]:
/tmp/mcp.sh "$MCP" tools/call '{"name":"run_code","arguments":{"code":"const p={}; for (const n of [\"Date\",\"fetch\",\"Math\",\"JSON\",\"Promise\",\"Map\",\"console\",\"process\",\"require\"]) { try { p[n]=typeof eval(n); } catch(e) { p[n]=\"MISSING\"; } } p"}}' \
  | python3 -c "
import json,sys
print(json.dumps(json.loads(json.load(sys.stdin)['result']['content'][0]['text'])['success'],indent=1))
"

No `Date` is why the agent keeps a local `today` tool. No `Map` is the kind of thing a
model reaches for by habit. Both are written down in the approved skill, which is the
only reason the run above worked first time.

### One more, to answer the heckle

Somebody will assume the program was written in advance. It was not, and the cheapest
way to prove it is to ask something you obviously did not plan for. It writes a
different program and still answers in two turns.

Name the repository in the question. Leave it out and the agent has nothing to work
from, which is why the approved skill tells it to ask rather than pick one.

In [ ]:
AGENT_PREFIX=prtriage $ASK "On $DEMO_REPO, of the open pull requests that are on hold, which was opened earliest? Answer in one line." \
  | tail -6

## 6. Take the write tools away

The report needs to read pull requests. The agent has been holding forty-five tools
all along, seventeen of which write. This policy names what it actually needs, on the
MCP method name, at the gateway.

In [ ]:
cat $PART8/yaml/20-authz-readonly.yaml

In [ ]:
$KC apply -f $PART8/yaml/20-authz-readonly.yaml
sleep 8
echo "  == a program that tries a denied operation =="
/tmp/mcp.sh "$MCP" tools/call '{"name":"run_code","arguments":{"code":"await merge_pull_request({ owner: \"kagent-dev\", repo: \"kagent\", pullNumber: 2790 })"}}' \
  | python3 -c "
import json,sys
print('  ',json.load(sys.stdin)['result']['content'][0]['text'])
"

`merge_pull_request is not defined`. The generated TypeScript API is built **after**
the policy is applied, so a denied operation is not a function in the sandbox at all.
The program cannot express the call.

Now ask the agent, which is what the room should see. Do this one in the kagent UI too.

In [ ]:
$PART8/scripts/reload-agent.sh
AGENT_PREFIX=prtriage $ASK "Merge pull request 2790 on kagent-dev/kagent right now. Use whatever tool you have."

The PAT in that Secret still has every permission it always had. **The gateway is what
makes this agent read-only, and it does it per agent**, which no token scope can.

And the report still works, on the tools it actually needs:

In [ ]:
AGENT_PREFIX=prtriage $ASK "Give me the release report for $DEMO_REPO, all open pull requests." | tail -16

## What that was

1. A developer built and shipped an agent from approved parts, and never held a
   credential for the system it talks to.
2. The same job went from nine round trips to two on one field, and from twenty to two
   against a live repo where it had also been getting the answer wrong, because the
   data it had to remember stopped passing through it.
3. Seventeen write tools were taken away per agent, and the code sandbox could not
   even name the denied one.

None of the three happened in the agent's code.

---

## Appendix: the other two settings

Skip this in a short slot. `toolMode` has four values, and they attack two different
problems:

- **`Search`** is discovery. 44 tools become `get_tool` and `invoke_tool`: names stay
  in `get_tool`'s description, schemas are fetched on demand. It cuts tool count but
  not round trips, because it still invokes one operation per call.
- **`Code`** is execution, which is what step 5 showed. It cuts round trips, and pays
  6,302 tokens to carry all 44 signatures in `run_code`'s description.
- **`CodeSearch`** is both: `get_tool` plus `run_code`, with the generated API dropped
  from the description and discovered instead. `invoke_tool` disappears entirely.

Measured on this cluster, same 8-PR question, `claude-haiku-4-5`, agentgateway
`v2026.8.2`:

| `toolMode` | tools held | schema tokens/turn | round trips | payload through model | secs |
|---|---|---|---|---|---|
| `Standard`   | 45 | 14,572 | 20 | 87,630 B | 39 |
| `Search`     | 3  | 986    | 22 | 48,216 B | 36 |
| `Code`       | 2  | 6,302  | 2  | 11 B     | 17 |
| `CodeSearch` | 3  | 1,300  | 2  | 11 B     | 18 |

`CodeSearch`, which step 5 used, is the cheapest of the four. It is also the most dependent on guidance,
because it withholds both the catalogue and the signatures, so the model has to get
the schemas either from `get_tool` (a turn each) or from the approved skill (free, and
versioned).

That dependency is not theoretical. Both settings that withhold information failed the
first time this ran, in the same way: the model guessed instead of asking. `Search`
sent `per_page` for `perPage` and folded the owner into `repo` as `"owner/repo"`, and
each wrong guess retried the largest call in the job, which is how it reached 222,000
bytes and came out worse than doing nothing. `CodeSearch` did the same and then wrote
an `else if` chain whose `Array.isArray(checks.check_runs)` guard swallowed the
approval test, reporting eight unapproved pull requests as ready to merge.

One line in the approved skill fixed both:

> Call `get_tool` for a tool before you first `invoke_tool` it, and use the argument
> names it gives back. Do not guess them.

So the choice is not really which setting is fastest. Pick `Code` when you want the
signatures always present and cannot rely on guidance. Pick `CodeSearch` when a
registry is feeding your agents versioned rules and you want the cheapest context.
Pick `Search` when a task touches one or two tools and a program is overkill.

Run the two step 5 did not:

In [ ]:
for m in Search Code; do
  $KC -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
    --type=merge -p "{\"spec\":{\"entMcp\":{\"toolMode\":\"$m\"}}}" >/dev/null
  sleep 8
  $PART8/scripts/reload-agent.sh >/dev/null
  echo "### $m"
  AGENT_PREFIX=prtriage $ASK "Give me the release report for $DEMO_REPO, all open pull requests." > /tmp/run-$m.txt 2>&1
  $PART8/scripts/trace-cost.sh /tmp/run-$m.txt
  grep -E "^Ready to merge" /tmp/run-$m.txt | head -1
done

## Reset / teardown

Back to a clean, re-runnable state. The Secret and backend stay, so `setup.sh` is a
one-off.

In [ ]:
$KC -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$KC -n agentgateway-system patch enterpriseagentgatewaybackend github-mcp \
  --type=merge -p '{"spec":{"entMcp":{"toolMode":"Standard"}}}'
$PART8/scripts/reload-agent.sh
echo "✓ back to Standard mode, no policy"

Remove Part 8 entirely:

In [ ]:
arctl delete deployment prtriage 2>/dev/null || true
arctl delete agent prtriage 2>/dev/null || true
arctl delete mcpserver github-mcp 2>/dev/null || true
arctl delete skill release-report 2>/dev/null || true
$KC -n agentgateway-system delete enterpriseagentgatewaypolicy github-readonly --ignore-not-found
$KC -n agentgateway-system delete enterpriseagentgatewaybackend github-mcp --ignore-not-found
$KC -n agentgateway-system delete httproute github-mcp --ignore-not-found
$KC -n agentgateway-system delete secret github-mcp-pat --ignore-not-found
echo "✓ Part 8 removed"